<a href="https://colab.research.google.com/github/arivine/data-science-2026/blob/main/Pertemuan_6_Muhammad_Arifin_230401010367.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###  Identitas Mahasiswa
* **Nama Lengkap:** Muhammad Arifin
* **NIM:** 230401010367
* **Kelas:** IF 401
* **Program Studi:** PJJ Informatika

In [1]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import (train_test_split)

import pandas as pd

In [2]:
df = pd.DataFrame({
'Gender': ['male', 'female', 'female', 'male', 'female'],
'Survived': [0, 1, 1, 0, 1]
})
# ── Label Encoding ────────────────────────────────────────────────
le = LabelEncoder()
# fit_transform: belajar pemetaan + langsung transformasi
df['Gender_enc'] = le.fit_transform(df['Gender'])
print(df)
# Hasil: female → 0, male → 1
# Lihat pemetaan kelas
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print('Mapping:', mapping) # {'female': 0, 'male': 1}
# Decode balik ke label asli
decoded = le.inverse_transform([0, 1, 0])
print(decoded) # ['female' 'male' 'female']

   Gender  Survived  Gender_enc
0    male         0           1
1  female         1           0
2  female         1           0
3    male         0           1
4  female         1           0
Mapping: {'female': np.int64(0), 'male': np.int64(1)}
['female' 'male' 'female']


In [3]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
  columns=['sex', 'embarked'],
  drop_first=True, # hindari dummy variable trap
  dtype=int)

print('Kolom setelah encoding:')
print(df.columns.tolist())
# ['pclass','age','sibsp','parch','fare','survived',
# 'sex_male','embarked_Q','embarked_S']


Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


In [3]:
df = pd.DataFrame({'City': ['Jakarta','Surabaya','Bandung','Jakarta','Surabaya']})
# ── Cara 1: Pandas get_dummies (paling mudah) ─────────────────────
df_ohe = pd.get_dummies(df,
columns=['City'],
drop_first=False, # True untuk hindari dummy variable trap
dtype=int) # hasilkan 0/1 bukan True/False
print(df_ohe)
# ── Cara 2: sklearn OneHotEncoder (untuk pipeline ML) ─────────────
enc = OneHotEncoder(sparse_output=False, drop='first')
X_enc = enc.fit_transform(df[['City']])
print('Nama fitur:', enc.get_feature_names_out())
# ── Cara 3: Dengan drop_first=True (menghindari Dummy Variable Trap)
df_ohe_drop = pd.get_dummies(df, columns=['City'],
drop_first=True, dtype=int)
# Hasilnya: City_Surabaya, City_Bandung (City_Jakarta dihapus)
# City_Jakarta dapat disimpulkan: jika dua kolom lain = 0, maka Jakarta

   City_Bandung  City_Jakarta  City_Surabaya
0             0             1              0
1             0             0              1
2             1             0              0
3             0             1              0
4             0             0              1
Nama fitur: ['City_Jakarta' 'City_Surabaya']


In [4]:
df = pd.DataFrame({
'Pendidikan': ['SMA', 'S1', 'SD', 'D3', 'S2', 'SMP'],
'Gaji_juta': [5, 12, 3, 8, 18, 4]
})
# Definisikan urutan kategori secara eksplisit
edu_order = [['SD', 'SMP', 'SMA', 'D3', 'S1', 'S2']]
enc = OrdinalEncoder(
categories=edu_order,
handle_unknown='use_encoded_value',
unknown_value=-1) # kategori baru → -1
df['Pendidikan_enc'] = enc.fit_transform(df[['Pendidikan']])
print(df.sort_values('Pendidikan_enc'))
# SD=0, SMP=1, SMA=2, D3=3, S1=4, S2=5

  Pendidikan  Gaji_juta  Pendidikan_enc
2         SD          3             0.0
5        SMP          4             1.0
0        SMA          5             2.0
3         D3          8             3.0
1         S1         12             4.0
4         S2         18             5.0


In [5]:
df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12] # dalam juta rupiah
})
scaler = MinMaxScaler(feature_range=(0, 1)) # default
# fit_transform: belajar min/max + transformasi data
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])
print('Min per fitur :', scaler.data_min_) # [25 5]
print('Max per fitur :', scaler.data_max_) # [55 35]
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_sc', 'Pendapatan_sc']).round(3))
# PENTING: gunakan .transform() saja pada data baru (test set!)
# X_test_scaled = scaler.transform(X_test)
# Balik ke skala asli
X_original = scaler.inverse_transform(X_scaled)

Min per fitur : [25.  5.]
Max per fitur : [55. 35.]

   Usia_sc  Pendapatan_sc
0    0.000          0.000
1    0.667          0.500
2    0.233          0.100
3    1.000          1.000
4    0.100          0.233


In [6]:
df = pd.DataFrame({
'Usia': [25, 45, 32, 55, 28],
'Pendapatan': [5, 20, 8, 35, 12]
})
scaler = StandardScaler()
X = df[['Usia', 'Pendapatan']]
X_scaled = scaler.fit_transform(X)
print('Mean per fitur:', scaler.mean_) # rata-rata setiap kolom
print('Scale per fitur:', scaler.scale_) # std dev setiap kolom
print()
print(pd.DataFrame(X_scaled,
columns=['Usia_z', 'Pend_z']).round(3))
# Contoh output:
# Usia_z Pend_z
# 0 -1.110 -0.784 (di bawah rata-rata)
# 3 1.533 1.568 (di atas rata-rata)
# Selalu .transform() saja pada test set
# X_test_z = scaler.transform(X_test)

Mean per fitur: [37. 16.]
Scale per fitur: [11.296017   10.75174404]

   Usia_z  Pend_z
0  -1.062  -1.023
1   0.708   0.372
2  -0.443  -0.744
3   1.593   1.767
4  -0.797  -0.372


In [7]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(df[['Usia', 'Pendapatan']])
# Gunakan saat dataset mengandung outlier yang tidak bisa dihapus

In [8]:
import pandas as pd, seaborn as sns
import matplotlib.pyplot as plt
df = sns.load_dataset('titanic')
# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))
# survived=0: ~61.6%, survived=1: ~38.4% — kelas tidak seimbang!

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [9]:
# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())
# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
print('Missing setelah handling:')
print(df.isnull().sum()) # Semua harus 0

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


In [10]:
# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
columns=['sex', 'embarked'],
drop_first=True, # hindari dummy variable trap
dtype=int)
print('Kolom setelah encoding:')
print(df.columns.tolist())
# ['pclass','age','sibsp','parch','fare','survived',
# 'sex_male','embarked_Q','embarked_S']

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


In [11]:
from sklearn.model_selection import train_test_split
X = df.drop('survived', axis=1)
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(
X, y,
test_size=0.2,
random_state=42,
stratify=y # proporsi kelas terjaga
)
print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


In [12]:
from sklearn.preprocessing import StandardScaler
# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) TIDAK perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']
scaler = StandardScaler()
# fit_transform pada training set (belajar μ dan σ dari sini)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
# transform saja pada test set (gunakan μ dan σ dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])
print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))
print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8), y_test : (179,)
